In [23]:
from dotenv import load_dotenv 
load_dotenv()
import os

In [3]:
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader


In [30]:
embeddings = OpenAIEmbeddings(model= "text-embedding-3-large")
model = ChatOpenAI(model = "gpt-4o-mini")

In [10]:
loader = PyPDFLoader("../attention.pdf")
docs = loader.load()
docs = docs[:8]

In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 100)
final_document = text_splitter.split_documents(docs)

In [24]:
ASTRA_DB_TOKEN        = os.getenv("ASTRA_DB_APPLICATION_TOKEN")
ASTRA_DB_API_ENDPOINT = os.getenv("ASTRA_DB_API_ENDPOINT")

In [26]:
from langchain_astradb import AstraDBVectorStore

db = AstraDBVectorStore(
    embedding = embeddings,
    collection_name = "pdf_query",
    token = ASTRA_DB_TOKEN,
    api_endpoint = ASTRA_DB_API_ENDPOINT

)

In [35]:
db.add_documents(final_document)  # add separately


['00e0472b26c94c3ab3218c9b000ba2db',
 '50773b957a6b4bbb95aee498b00cf830',
 '8dee9208076940b0ab5b6176990ab597',
 'fa15df8d855b4393b376fb81091dcf3e',
 'b6e60ba53af044bd955bfb297443a778',
 'bf2310215b5d4cf897c563b890412551',
 '1e53e2d4410d4512994e2d4fa6af88b5',
 '75ce11e3062e42389dda504cf7ab5a57',
 '417da24fae2c43dfab9d8f3f16ec496c',
 '360600c58bee403a803cc088c07f9c6c',
 '9a885ba514084bd982512ccced3c87ab',
 '8b3e280848944a0fb2c540ddc8cdc3ba',
 'b728cc362dba40639734bc1f27d3e48a',
 '29142a35f970439b86fee243a8e40157',
 'c109319cc9e448f08d71dcc62cd3efcb',
 '118ec7a0da784f85a0a1fc0da8188bbe',
 '113e049f267e44fc86a1a6a3e9c9bddb',
 '28811370e38d4ef5a407edf31b639a4e',
 'd8a76627fe874c55966b54a484f8dbfa',
 'af8c98482c72444596e937120af4e054',
 'bed1ec13ec414a0a8a011efbd3bfe845',
 'b02549a93441456f922c2549c2047173',
 '930dfa5696b5460aba6b365df44d7ded',
 '67c5583546d34061b29b2c79da04f61c',
 '78cb2d9011b04513ae3d4d80bc026ef8',
 '6e1b537755184360979074499ab80789',
 'b08ad36a2939404490691caa91ae32d9',
 

In [36]:
retriever = db.as_retriever()

In [37]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [38]:
prompt = ChatPromptTemplate.from_template("""
Answer the question based only on the context below.

<context>
{context}
</context>

Question: {input}
""")

In [39]:
# Step 1: chain that stuffs docs into the prompt
combine_docs_chain = create_stuff_documents_chain(model, prompt)

# Step 2: retrieval chain that fetches + answers
chain = create_retrieval_chain(retriever, combine_docs_chain)


In [40]:
result = chain.invoke({"input": "What is attention mechanism?"})

print(result["answer"])          # the answer
print(result["context"])  

An attention mechanism is a method that relates different positions of a single sequence to compute a representation of that sequence. Self-attention, a specific type of attention mechanism, utilizes this concept to allow every position in the sequence to attend to all other positions, which has been successfully applied in various tasks such as reading comprehension and language modeling.
[Document(id='75ce11e3062e42389dda504cf7ab5a57', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '../attention.pdf', 'total_pages': 15, 'page': 1, 'page_label': '2'}, page_content='described in section 3.2.\nSelf-attention, sometimes called intra-attention is an attention mechanism relating differ